# 03 — DPO with UltraFeedback (Condition A)

**Notebook 3 of 7 (seed-parameterized — run 3 times).** Trains DPO on 500 general-purpose preference pairs from `HuggingFaceH4/ultrafeedback_binarized`, then evaluates on GSM8K[:500].

This is the **control** condition: the "did DPO on standard chat preferences move the needle on reasoning?" question. Compare against Notebook 04 (self-generated reasoning preferences) to isolate the effect of the preference *signal*.

- **Runtime:** ~30 min train + ~1.5h eval ≈ **2 hours per seed** on Colab T4.
- **Total across 3 seeds:** ~6 hours (three separate sessions).
- **Outputs:** `results/dpo_A_seed{1,2,3}.json` on Drive.

## 1. Install dependencies

Same stack as Notebooks 01 and 02. If you've already run those in this Colab session, the install below is a no-op.

In [ ]:
%%capture
!pip install -q unsloth
!pip install -q --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

## 2. Configuration — **set `SEED` for this run**

You will run this notebook three times, changing `SEED` to 1, then 2, then 3. Each run produces `results/dpo_A_seed{SEED}.json` on Drive. The three seeded runs together give the sample size we need for the aggregated statistics in Notebook 05.

In [ ]:
SEED = 1   # <-- change to 2 for the second run, 3 for the third

# Sanity: fail loudly on typos
assert SEED in (1, 2, 3), f"Expected SEED in (1,2,3), got {SEED}"

## 3. Mount Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/dpo-reasoning-transfer'
os.makedirs(f'{PROJECT_ROOT}/results', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/adapters', exist_ok=True)

RESULTS_PATH  = f'{PROJECT_ROOT}/results/dpo_A_seed{SEED}.json'
ADAPTER_PATH  = f'{PROJECT_ROOT}/adapters/dpo_A_seed{SEED}'
BASELINE_PATH = f'{PROJECT_ROOT}/results/baseline_gsm8k_500.json'

print(f'Project root: {PROJECT_ROOT}')
print(f'This run will write:')
print(f'  results: {RESULTS_PATH}')
print(f'  adapter: {ADAPTER_PATH}   # loaded by Notebook 06')

## 4. Load `src/`

In [ ]:
import sys
CODE_DIR = PROJECT_ROOT   # or f'{PROJECT_ROOT}/repo' if cloned from GitHub
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from src.train import attach_lora, train_dpo_and_evaluate, compare_to_baseline
print('src/train.py loaded')

## 5. Load the base model and attach LoRA adapters

Same base model as Notebooks 01 and 02. LoRA config (r=16, α=16, standard attention + MLP targets) is identical between Conditions A and B — the *only* variable between them is the preference dataset.

In [ ]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Llama-3.2-3B-Instruct',
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

model = attach_lora(model, r=16, lora_alpha=16, seed=SEED)
print(f'Model + LoRA attached (LoRA seed = {SEED})')

## 6. Load and format UltraFeedback

Shuffle the full `train_prefs` split with this run's `SEED`, then take the first 500 pairs. Different seeds therefore see different 500-pair slices — this deliberately absorbs data-sampling variance into the seed effect, alongside the initialization variance.

In [ ]:
from datasets import load_dataset
from src.prefs import format_ultrafeedback_example

raw = load_dataset('HuggingFaceH4/ultrafeedback_binarized', split='train_prefs')
raw = raw.shuffle(seed=SEED).select(range(500))
print(f'Loaded {len(raw)} UltraFeedback pairs (seed={SEED})')

# Apply chat template + extract chosen/rejected strings
dpo_dataset = raw.map(
    lambda ex: format_ultrafeedback_example(ex, tokenizer),
    remove_columns=raw.column_names,
    desc='Formatting for DPOTrainer',
)
print(f'Formatted columns: {dpo_dataset.column_names}')
print()
print('--- Example prompt (first 300 chars) ---')
print(dpo_dataset[0]['prompt'][:300])
print()
print('--- Example chosen  (first 200 chars) ---')
print(dpo_dataset[0]['chosen'][:200])
print()
print('--- Example rejected (first 200 chars) ---')
print(dpo_dataset[0]['rejected'][:200])

## 7. Train + evaluate

One call. Skips training and eval if `RESULTS_PATH` already exists for this (condition, seed) tuple. If Colab drops during the eval half, re-running this cell resumes from the partial file.

In [ ]:
results = train_dpo_and_evaluate(
    model=model,
    tokenizer=tokenizer,
    dpo_dataset=dpo_dataset,
    results_output_path=RESULTS_PATH,
    condition_name='dpo_A_ultrafeedback',
    preference_dataset_name='HuggingFaceH4/ultrafeedback_binarized:train_prefs[shuffle(SEED)][:500]',
    seed=SEED,
    n_eval_samples=500,
    output_dir=f'/tmp/dpo_A_seed{SEED}',
    adapter_save_path=ADAPTER_PATH,
)
print()
print('=' * 60)
print(f"Condition A, seed {SEED}: {results['accuracy']:.1%} "
      f"({results['n_correct']}/{results['n_samples']})")
print('=' * 60)

## 8. Quick comparison to baseline

A fast preview of the delta vs. baseline. **This is not the final analysis** — Notebook 05 pools across all three seeds and both conditions with proper multiple-comparison correction. But the single-seed McNemar p here is a useful reality check that the training didn't obviously break anything.

In [ ]:
import json

with open(BASELINE_PATH) as f:
    baseline_results = json.load(f)

cmp = compare_to_baseline(baseline_results, results)

print(f'Baseline accuracy:  {cmp["baseline_accuracy"]:.1%}')
print(f'Condition A (seed {SEED}):     {cmp["dpo_accuracy"]:.1%}')
print(f'Delta:              {cmp["delta_pp"]:+.1f} pp')
print()
print(f'Discordant pairs (out of {cmp["n_compared"]}):')
print(f'  baseline right, DPO wrong: {cmp["baseline_right_dpo_wrong"]}')
print(f'  baseline wrong, DPO right: {cmp["baseline_wrong_dpo_right"]}')
print()
print(f'Single-seed McNemar p (exact, two-tailed): {cmp["mcnemar_p"]:.4f}')
print()
print('Reminder: a single seed is underpowered; Notebook 05 aggregates seeds 1–3.')

## 9. Training-loss trace

Sanity check on the training itself — did DPO loss actually decrease, and did the reward margin (chosen − rejected implicit reward) go positive?

In [ ]:
import matplotlib.pyplot as plt

log_history = results['run_metadata']['training']['log_history']
losses  = [(e['step'], e['loss'])              for e in log_history if 'loss' in e]
margins = [(e['step'], e['rewards/margins'])   for e in log_history if 'rewards/margins' in e]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

if losses:
    xs, ys = zip(*losses)
    axes[0].plot(xs, ys, marker='.', color='#4C72B0')
    axes[0].set_xlabel('Training step')
    axes[0].set_ylabel('DPO loss')
    axes[0].set_title(f'Training loss (seed {SEED})')
    axes[0].grid(alpha=0.3)

if margins:
    xs, ys = zip(*margins)
    axes[1].plot(xs, ys, marker='.', color='#DD8452')
    axes[1].axhline(0, color='k', ls='--', lw=1, alpha=0.5)
    axes[1].set_xlabel('Training step')
    axes[1].set_ylabel('rewards/margins (chosen − rejected)')
    axes[1].set_title(f'Implicit reward margin (seed {SEED})')
    axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Next

- If this was seed 1: change `SEED` to 2 at the top and re-run cells 5 onward.
- After all three seeds: run **Notebook 04** for Condition B (self-generated reasoning preferences), also three times.
- With all six DPO runs (A×{1,2,3} + B×{1,2,3}) plus baseline cached in `results/`, Notebook 05 does the aggregated statistics.